# CeNN V5.3 × All Six Qwen3.5 Full-Attention Layers

**Progressive multi-layer replacement • checkpointed training • automatic statistics/log download**

This experiment progressively replaces **all six full-attention layers** of `Qwen/Qwen3.5-0.8B` (`[3, 7, 11, 15, 19, 23]`) with V5.3 CeNN modules. The other **18 gated-delta/linear-attention layers remain original**, because this adapter is not compatible with their interfaces.

- `REPLACEMENT_MODE='one_group'`: four of eight Q heads per full-attention layer use V5.3; four native SDPA heads are retained.
- `REPLACEMENT_MODE='both_groups'`: all eight Q heads per full-attention layer use V5.3; there is no exact-attention group in those six layers.
- `FORCE_ALL_SIX=True` performs the complete six-stage experiment **even if an intermediate quality gate fails**. A diagnostic result is not a validated faster deployable model.
- Training logs, measurements, adapter-only checkpoints, run manifest, diagnostic report and charts are packaged automatically; the last cell triggers **Google Colab's file download**, including if the training/audit phase raises an exception.

**Limitations:** unpadded text-only full-prefix inference, `use_cache=False`; no Qwen hybrid KV cache, no `.generate()` decoding, no fused recurrent GPU kernel. Native Qwen may remain faster. Run `Runtime → Change runtime type → GPU` before `Run all`.

## 0 — Install and validate repository code

Import canonical source from the GitHub repository. No user uploads or Drive connection are needed. A pinned Transformers release is used to protect the Qwen3.5 adapter signature.

In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"


In [ ]:
import os, sys, subprocess, gc, json, hashlib, math, random, time, csv, zipfile, traceback
from pathlib import Path
from contextlib import nullcontext
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
        'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))
from tinycenn_lm.cenn_v53_multilayer import (
    FULL_ATTENTION_LAYERS, FullAttentionGraftManager,
    validate_full_attention_targets, check_quality, experiment_fingerprint, append_jsonl)
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3
from tinycenn_lm.cenn_head_rescue_v5 import Qwen35HeadRescueV5
from tinycenn_lm.cenn_head_rescue_v51 import CeNNQueryRescueV51, warmstart_v5_delta_core
from tinycenn_lm.cenn_head_rescue_v52 import CeNNHeadRescueV52, warmstart_v51_to_v52
from tinycenn_lm.cenn_head_rescue_v53 import CeNNHeadRescueV53, warmstart_v52_to_v53
from transformers import AutoConfig, AutoTokenizer, Qwen3_5ForCausalLM
from huggingface_hub import model_info
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
SOURCE_FILES=[REPO/'src/tinycenn_lm'/f for f in ('cenn_v53_multilayer.py',
    'cenn_deltacell_v3.py','cenn_head_rescue_v5.py','cenn_head_rescue_v51.py',
    'cenn_head_rescue_v52.py','cenn_head_rescue_v53.py')]
SOURCE_SHA256=hashlib.sha256(b''.join(p.read_bytes() for p in SOURCE_FILES)).hexdigest()
subprocess.run([sys.executable,'-m','pytest','-q',
    str(REPO/'tests/test_cenn_v53_multilayer.py'),
    str(REPO/'tests/test_cenn_head_rescue_v53.py')],check=True)
assert torch.cuda.is_available(), 'NVIDIA CUDA GPU required: Runtime → Change runtime type → GPU'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('Git',GIT_SHA, 'source SHA256',SOURCE_SHA256)
print('Torch',torch.__version__,'GPU',torch.cuda.get_device_name(0),'dtype',DTYPE)


## 1 — Experiment controls

**Quick** is a debugging/feasibility pass, not a publishable benchmark. **Research** increases training steps, validation examples and sequence length. For a genuine **all-attention-heads substitution**, choose `both_groups`; both modes replace all **six eligible layers**.

In [ ]:
# @title Multi-layer V5.3 experiment configuration
PROFILE='quick' # @param ['quick','research']
REPLACEMENT_MODE='one_group' # @param ['one_group','both_groups']
FORCE_ALL_SIX=True # @param {type:'boolean'}
AUTO_DOWNLOAD_RESULTS=True # @param {type:'boolean'}
MODEL_ID='Qwen/Qwen3.5-0.8B'
TARGET_LAYERS=list(FULL_ATTENTION_LAYERS)
SEQ_LEN=128 if PROFILE=='quick' else 256
TRAIN_BLOCKS=32 if PROFILE=='quick' else 96
DEV_BLOCKS=16 if PROFILE=='quick' else 48
AUDIT_BLOCKS=16 if PROFILE=='quick' else 48
AUDIT_START_TOKEN=1_600_000 # disjoint WikiText-2 TRAIN tail, not independent external test
BATCH_SIZE=2
BASELINE_STEPS=60 if PROFILE=='quick' else 250
V53_STEPS=100 if PROFILE=='quick' else 500
BASELINE_LR=2e-3
V53_LR=6e-4
FEATURE_DIM=64
LOCAL_WINDOW=128
CHUNK_SIZE=32
RANK=16
SEED=5306
LOG_EVERY=10 if PROFILE=='quick' else 25
MIN_COSINE=.95
MIN_AGREEMENT=.98
MAX_DELTA_NLL=.10
MIN_SPEEDUP=1.05
OUT=Path('/content/qwen35_cenn_v53_all6_results')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
TRAIN_EVENTS=OUT/'training_steps.jsonl'
TEXT_LOG=OUT/'training.log'
STAGE_CSV=OUT/'cumulative_layer_results.csv'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Targets:',TARGET_LAYERS,'mode:',REPLACEMENT_MODE,'force all six:',FORCE_ALL_SIX)


## 2 — Load the original frozen Qwen backbone

Qwen3.5-0.8B has 24 layers. The six full-attention layers are verified from model configuration before any modification, and the original attention objects are preserved for exact teacher outputs and native paired benchmarks.

In [ ]:
REVISION=model_info(MODEL_ID).sha
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
assert cfg.model_type=='qwen3_5_text' and cfg.hidden_size==1024 and cfg.num_hidden_layers==24
assert tuple(TARGET_LAYERS)==validate_full_attention_targets(cfg.layer_types, TARGET_LAYERS)
assert tuple(TARGET_LAYERS)==FULL_ATTENTION_LAYERS
TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,dtype=DTYPE,
    attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False)
model.config.use_cache=False
LAYERS=model.model.layers
MANAGER=FullAttentionGraftManager(LAYERS,TARGET_LAYERS)
HD=int(MANAGER.native[11].head_dim)
QHEADS=int(MANAGER.native[11].q_proj.out_features//(2*HD))
KVHEADS=int(MANAGER.native[11].k_proj.out_features//HD)
assert (QHEADS,KVHEADS,HD)==(8,2,256)
for i in TARGET_LAYERS:
    native=MANAGER.native[i]
    assert (native.q_proj.out_features//(2*native.head_dim),
            native.k_proj.out_features//native.head_dim, native.head_dim)==(8,2,256)
EXACT_GROUPS=(0,) if REPLACEMENT_MODE=='one_group' else ()
APPROX_KV=KVHEADS-len(EXACT_GROUPS)
APPROX_Q=APPROX_KV*(QHEADS//KVHEADS)
assert (APPROX_Q,APPROX_KV) in ((4,1),(8,2))
print('Pinned Qwen SHA:',REVISION,'24 layers; six full-attention targets:',TARGET_LAYERS)
print('Per target: native exact Q heads:',QHEADS-APPROX_Q, 'CeNN Q heads:',APPROX_Q)
# Fail fast if Transformers/Qwen implementation drift changes native parity.
probe=TOKENIZER('CeNN recurrent memory is tested against original Qwen attention.',
                add_special_tokens=False,return_tensors='pt').input_ids.to(DEVICE)
oracle=Qwen35HeadRescueV5(MANAGER.native[TARGET_LAYERS[0]],None,
                          exact_groups=tuple(range(KVHEADS))).eval()
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    native_probe=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
    with MANAGER.candidate(TARGET_LAYERS[0],oracle):
        exact_probe=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
max_native_diff=float((native_probe-exact_probe).abs().max())
assert max_native_diff<.05 and bool((native_probe.argmax(-1)==exact_probe.argmax(-1)).all()), max_native_diff
print('All-native SDPA wrapper parity:',max_native_diff)
del oracle,native_probe,exact_probe,probe;torch.cuda.empty_cache()


## 3 — Fixed train/validation data and native teacher

The **WikiText-2 test split is not used**. Training takes the earliest WikiText-2 train windows; validation uses the validation split. The final diagnostic uses a later, disjoint region of the *train* split, accessed only after all stages. This audit is not an independent benchmark.

In [ ]:
from datasets import load_dataset

def token_windows(split,count,length,start_token=0):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    needed=start_token+count*length
    ids=[]
    for row in ds:
        text=row['text'].strip()
        if not text: continue
        ids.extend(TOKENIZER.encode(text,add_special_tokens=False))
        ids.append(TOKENIZER.eos_token_id)
        if len(ids)>=needed: break
    if len(ids)<needed:
        raise RuntimeError(f'{split} contains {len(ids)} tokens; need {needed}. '
                           'Choose a lower AUDIT_START_TOKEN if the corpus changed.')
    return torch.tensor(ids[start_token:needed],dtype=torch.long).reshape(count,length)
TRAIN=token_windows('train',TRAIN_BLOCKS,SEQ_LEN)
DEV=token_windows('validation',DEV_BLOCKS,SEQ_LEN)
DATA_HASHES={k:hashlib.sha256(t.numpy().tobytes()).hexdigest()
             for k,t in (('train',TRAIN),('validation',DEV))}
assert TRAIN_BLOCKS*SEQ_LEN<AUDIT_START_TOKEN
CONFIG=dict(model=MODEL_ID,revision=REVISION,repo_sha=GIT_SHA,source_sha256=SOURCE_SHA256,
    profile=PROFILE,mode=REPLACEMENT_MODE,force_all_six=FORCE_ALL_SIX,
    targets=TARGET_LAYERS,seq_len=SEQ_LEN,train_blocks=TRAIN_BLOCKS,
    dev_blocks=DEV_BLOCKS,audit_blocks=AUDIT_BLOCKS,audit_offset=AUDIT_START_TOKEN,
    batch_size=BATCH_SIZE,baseline_steps=BASELINE_STEPS,v53_steps=V53_STEPS,
    baseline_lr=BASELINE_LR,v53_lr=V53_LR,feature_dim=FEATURE_DIM,
    local_window=LOCAL_WINDOW,chunk_size=CHUNK_SIZE,rank=RANK,seed=SEED,
    data_hashes=DATA_HASHES,gpu=torch.cuda.get_device_name(0),torch=torch.__version__)
RUN_FINGERPRINT=experiment_fingerprint(CONFIG)
(OUT/'config.json').write_text(json.dumps(CONFIG,indent=2,allow_nan=False))
print('Train/validation:',TRAIN.shape,DEV.shape,'fingerprint:',RUN_FINGERPRINT)


## 4 — Reproducible evaluation, teacher capture and training logging

**Sequential teacher capture** uses the original target attention module on hidden states produced by the **already-installed CeNN prefix**. This trains each subsequent layer for the distribution it will actually encounter. Validation uses full-model native teacher top-1 decisions and next-token NLL; timing compares native and cumulative modified models in alternating order.

In [ ]:
def log_line(message):
    line=f'[{time.strftime("%Y-%m-%dT%H:%M:%S")}] {message}'
    print(line,flush=True)
    with TEXT_LOG.open('a',encoding='utf-8') as f:f.write(line+'\n')

@torch.inference_mode()
def inference(blocks,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=blocks.to(DEVICE),use_cache=False,logits_to_keep=keep)

@torch.inference_mode()
def capture_target(idx,split,blocks):
    assert LAYERS[idx].self_attn is MANAGER.native[idx]
    stored={key:[] for key in ('x','y','cos','sin')}
    def hook(mod,args,kwargs,out):
        x=kwargs.get('hidden_states')
        if x is None and args:x=args[0]
        assert x is not None
        y=out[0] if isinstance(out,tuple) else out
        cos,sin=kwargs['position_embeddings']
        stored['x'].append(x.detach().to('cpu',dtype=torch.float16))
        stored['y'].append(y.detach().to('cpu',dtype=torch.float16))
        stored['cos'].append(cos.detach().cpu().float())
        stored['sin'].append(sin.detach().cpu().float())
    handle=MANAGER.native[idx].register_forward_hook(hook,with_kwargs=True)
    try:
        for i in range(0,len(blocks),BATCH_SIZE):
            inference(blocks[i:i+BATCH_SIZE])
    finally:
        handle.remove()
    answer={key:torch.cat(val,dim=0).contiguous() for key,val in stored.items()}
    assert answer['x'].shape[:2]==blocks.shape
    return answer

def batch_from_capture(data,indices):
    return (data['x'][indices].to(DEVICE,dtype=DTYPE),
            data['y'][indices].to(DEVICE,dtype=DTYPE),
            (data['cos'][indices].to(DEVICE),data['sin'][indices].to(DEVICE)))

def mixer_loss(pred,ref):
    p=pred.float();r=ref.float()
    nmse=(p-r).square().mean()/r.square().mean().clamp_min(1.e-8)
    cosine=F.cosine_similarity(p,r,dim=-1).mean()
    return nmse+.2*(1.-cosine)

@torch.inference_mode()
def local_validation(adapter,data):
    adapter.eval(); err=energy=cs=0.;count=0
    for start in range(0,len(data['x']),BATCH_SIZE):
        idx=torch.arange(start,min(start+BATCH_SIZE,len(data['x'])))
        x,y,rope=batch_from_capture(data,idx)
        with torch.autocast('cuda',dtype=DTYPE):out,_=adapter(x,position_embeddings=rope)
        pp=out.float();yy=y.float()
        err+=(pp-yy).square().sum().item()
        energy+=yy.square().sum().item()
        cs+=F.cosine_similarity(pp,yy,dim=-1).sum().item()
        count+=pp.shape[0]*pp.shape[1]
    return dict(nmse=err/max(energy,1.e-8),cosine=cs/max(count,1))

@torch.inference_mode()
def model_metrics(blocks,teacher_top1=None):
    count=0;nll=0.;correct=0;agreement=0;top1=[]
    for start in range(0,len(blocks),BATCH_SIZE):
        chunk=blocks[start:start+BATCH_SIZE]
        positions=torch.arange(chunk.shape[1]-1,device=DEVICE)
        logits=inference(chunk,keep=positions).logits.float()
        gold=chunk[:,1:].to(DEVICE)
        logp=F.log_softmax(logits,dim=-1)
        nll-=logp.gather(-1,gold.unsqueeze(-1)).sum().item()
        pred=logits.argmax(-1)
        correct+=(pred==gold).sum().item();count+=gold.numel()
        if teacher_top1 is None:
            top1.append(pred.cpu().to(torch.int32))
        else:
            ref=teacher_top1[start//BATCH_SIZE].to(DEVICE)
            assert ref.shape==pred.shape
            agreement+=(pred==ref).sum().item()
    result=dict(nll=nll/count,accuracy=correct/count,decisions=count)
    if teacher_top1 is not None:result['teacher_agreement']=agreement/count
    return result,top1

@torch.inference_mode()
def paired_prefill(blocks,warmups=2,repeats=6):
    sample=blocks[:min(len(blocks),BATCH_SIZE)].to(DEVICE)
    native_times=[];student_times=[]
    def one(native):
        with MANAGER.native_context() if native else nullcontext():
            begin=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
            begin.record()
            logits=inference(sample,keep=1).logits
            end.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(logits).all())
            return begin.elapsed_time(end)
    for i in range(warmups+repeats):
        if i%2: st=one(False);nt=one(True)
        else: nt=one(True);st=one(False)
        if i>=warmups:native_times.append(nt);student_times.append(st)
    return dict(native_ms=float(np.median(native_times)),
                student_ms=float(np.median(student_times)),
                speedup=float(np.median(native_times)/np.median(student_times)),
                native_iqr_ms=float(np.percentile(native_times,75)-np.percentile(native_times,25)),
                student_iqr_ms=float(np.percentile(student_times,75)-np.percentile(student_times,25)),
                repeats=repeats)


## 5 — Construct and train V5.3 modules

Each stage trains a V3 Delta + Local128 base, then strictly transfers its fitted memory parameters to V5.1 → V5.2 → V5.3. New correction paths are **initialized as identity**, so the V5.3 starting point matches the trained baseline. All original Qwen parameters remain frozen; only the CeNN core receives gradients. Every optimizer step is written immediately to `training_steps.jsonl`.

In [ ]:
def make_base(idx):
    core=CeNNDeltaCellV3(APPROX_Q,APPROX_KV,HD,
        feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE,mode='none',
        local_window=LOCAL_WINDOW).to(DEVICE)
    return Qwen35HeadRescueV5(MANAGER.native[idx],core,
                              exact_groups=EXACT_GROUPS).eval()

def make_v53(idx,base_state):
    common=dict(num_heads=APPROX_Q,num_kv_heads=APPROX_KV,head_dim=HD,
                feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE,
                local_window=LOCAL_WINDOW,correction_rank=RANK,
                query_router=True,query_correction=True)
    v51=CeNNQueryRescueV51(**common)
    warmstart_v5_delta_core(v51,base_state)
    v52=CeNNHeadRescueV52(**common,post_rank=RANK,post_correction=True)
    warmstart_v51_to_v52(v52,v51.state_dict())
    v53=CeNNHeadRescueV53(**common,post_rank=RANK,post_correction=True,
          cross_rank=RANK,cross_correction=True,output_calibration=True)
    warmstart_v52_to_v53(v53,v52.state_dict())
    return Qwen35HeadRescueV5(MANAGER.native[idx],v53.to(DEVICE),
                              exact_groups=EXACT_GROUPS).eval()

def fit_local(idx,stage,adapter,train,dev,steps,lr):
    core=adapter.core
    core.requires_grad_(True)
    optimizer=torch.optim.AdamW(core.parameters(),lr=lr,weight_decay=.005)
    schedule=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=steps,eta_min=lr*.1)
    scaler=torch.amp.GradScaler('cuda',enabled=DTYPE==torch.float16)
    generator=torch.Generator().manual_seed(SEED+idx*29+(0 if stage=='delta' else 5000))
    indices=torch.randint(0,len(train['x']),(steps,BATCH_SIZE),generator=generator)
    best=math.inf;best_sd=None;best_stats=None
    start_time=time.perf_counter()
    for step in range(1,steps+1):
        ix=indices[step-1];x,y,rope=batch_from_capture(train,ix)
        core.train();optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
            loss=mixer_loss(pred,y)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'L{idx}/{stage} step {step}: nonfinite loss')
        scaler.scale(loss).backward();scaler.unscale_(optimizer)
        norm=torch.nn.utils.clip_grad_norm_(core.parameters(),1.)
        scaler.step(optimizer);scaler.update();schedule.step()
        event=dict(layer=idx,stage=stage,step=step,total_steps=steps,
                   loss=float(loss.detach()),grad_norm=float(norm),
                   lr=float(optimizer.param_groups[0]['lr']),
                   elapsed_seconds=float(time.perf_counter()-start_time))
        if step==1 or step==steps or step%LOG_EVERY==0:
            stats=local_validation(adapter,dev)
            event.update(val_cosine=stats['cosine'],val_nmse=stats['nmse'])
            score=stats['nmse']+.2*(1-stats['cosine'])
            if score<best:
                best=score;best_stats=stats
                best_sd={k:v.detach().cpu().clone() for k,v in core.state_dict().items()}
            log_line(f'L{idx} {stage} {step}/{steps}: loss={event["loss"]:.5f} '
                     f'cos={stats["cosine"]:.5f} nmse={stats["nmse"]:.5f}')
        append_jsonl(TRAIN_EVENTS,event)
    assert best_sd is not None
    core.load_state_dict(best_sd,strict=True);adapter.eval()
    core.requires_grad_(False)
    return best_sd,best_stats

@torch.inference_mode()
def assert_warmstart_parity(base,v53,dev):
    x,_,rope=batch_from_capture(dev,torch.tensor([0]))
    with torch.autocast('cuda',dtype=DTYPE):
        old,_=base(x,position_embeddings=rope)
        new,_=v53(x,position_embeddings=rope)
    max_diff=float((old-new).abs().max())
    assert max_diff<.01,f'Non-neutral warm-start difference: {max_diff}'
    return max_diff


## 6 — Run six progressive substitutions, evaluate, and **automatically download results**

The entire training/audit phase runs inside `try/finally`: even if a stage fails, its partial training logs and completed checkpoints are zipped and the browser download is requested. The archive contains `training.log`, `training_steps.jsonl`, `cumulative_layer_results.csv`, six per-layer adapter checkpoints when training succeeds, `final_audit.json`, `config.json`, a chart and an artifact manifest.

**The notebook does not mark the six-layer model successful unless both quality and speed checks pass.** If `FORCE_ALL_SIX=True`, layers continue to be installed for research regardless of intermediate failures.

In [ ]:
REPORT=[]
RUN_OK=False
RUN_ERROR=None
RUN_START=time.time()
NATIVE_DEV=None
FINAL=None
ARCHIVE=None
try:
    log_line('START V5.3 six-layer progressive substitution '+RUN_FINGERPRINT[:16])
    with MANAGER.native_context():
        NATIVE_DEV,NATIVE_TOP1=model_metrics(DEV)
    log_line('NATIVE validation '+json.dumps(NATIVE_DEV))
    for layer in TARGET_LAYERS:
        log_line(f'BEGIN layer {layer}; installed prefix {MANAGER.installed_layers}')
        # Native target sees the *already replaced* preceding layers.
        captured_train=capture_target(layer,'train',TRAIN)
        captured_dev=capture_target(layer,'dev',DEV)
        torch.manual_seed(SEED+layer*13)
        baseline=make_base(layer)
        baseline_state,baseline_local=fit_local(layer,'delta',baseline,
            captured_train,captured_dev,BASELINE_STEPS,BASELINE_LR)
        torch.save(dict(core_state=baseline_state,layer=layer,
                        stage='delta',fingerprint=RUN_FINGERPRINT,
                        validation=baseline_local),
            OUT/'checkpoints'/f'layer{layer:02d}_delta_seed.pt')
        adapter=make_v53(layer,baseline_state)
        initial_diff=assert_warmstart_parity(baseline,adapter,captured_dev)
        del baseline
        torch.cuda.empty_cache()
        trained_state,local=fit_local(layer,'v53',adapter,
            captured_train,captured_dev,V53_STEPS,V53_LR)
        checkpoint=OUT/'checkpoints'/f'layer{layer:02d}_v53_core.pt'
        torch.save(dict(core_state=trained_state,layer=layer,exact_groups=EXACT_GROUPS,
                        core_config=adapter.core.config,validation=local,
                        mode=REPLACEMENT_MODE,fingerprint=RUN_FINGERPRINT,
                        model_revision=REVISION,repo_sha=GIT_SHA),checkpoint)
        # A candidate always measures the cumulative (already-installed) model.
        with MANAGER.candidate(layer,adapter):
            student_dev,_=model_metrics(DEV,teacher_top1=NATIVE_TOP1)
            latency=paired_prefill(DEV,warmups=2,repeats=6 if PROFILE=='quick' else 15)
        delta_nll=student_dev['nll']-NATIVE_DEV['nll']
        gates=check_quality(local['cosine'],student_dev['teacher_agreement'],
                            delta_nll,latency['speedup'],
                            min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
                            max_delta_nll=MAX_DELTA_NLL,min_speedup=MIN_SPEEDUP)
        should_install=bool(FORCE_ALL_SIX or gates.quality_pass)
        if should_install:
            MANAGER.install(layer,adapter)
        row=dict(layer=layer,installed=should_install,forced=bool(should_install and not gates.quality_pass),
                 installed_count=len(MANAGER.installed_layers),
                 exact_query_heads_per_layer=QHEADS-APPROX_Q,cenn_query_heads_per_layer=APPROX_Q,
                 warmstart_max_abs_diff=initial_diff,
                 local_cosine=local['cosine'],local_nmse=local['nmse'],
                 cumulative_dev_nll=student_dev['nll'],dev_delta_nll=delta_nll,
                 cumulative_dev_agreement=student_dev['teacher_agreement'],
                 cumulative_dev_accuracy=student_dev['accuracy'],
                 native_ms=latency['native_ms'],cumulative_student_ms=latency['student_ms'],
                 cumulative_speedup=latency['speedup'],
                 quality_pass=gates.quality_pass,deploy_pass=gates.deploy_pass,
                 train_delta_steps=BASELINE_STEPS,train_v53_steps=V53_STEPS,
                 trainable_core_params=sum(p.numel() for p in adapter.core.parameters()))
        REPORT.append(row)
        pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
        log_line('LAYER SUMMARY '+json.dumps(row,allow_nan=False))
        del captured_train,captured_dev,trained_state,baseline_state
        gc.collect();torch.cuda.empty_cache()
    # Once all planned stages finish, compute native-vs-cumulative results.
    AUDIT=token_windows('train',AUDIT_BLOCKS,SEQ_LEN,start_token=AUDIT_START_TOKEN)
    DATA_HASHES['diagnostic_train_tail']=hashlib.sha256(AUDIT.numpy().tobytes()).hexdigest()
    assert len(set(DATA_HASHES.values()))==3
    with MANAGER.native_context():
        NATIVE_AUDIT,AUDIT_TOP1=model_metrics(AUDIT)
    STUDENT_AUDIT,_=model_metrics(AUDIT,teacher_top1=AUDIT_TOP1)
    FINAL_TIMING=paired_prefill(AUDIT,warmups=5,repeats=20)
    all_six=len(MANAGER.installed_layers)==len(TARGET_LAYERS)
    cumulative_delta=STUDENT_AUDIT['nll']-NATIVE_AUDIT['nll']
    stage_quality=all(row['quality_pass'] for row in REPORT if row['installed'])
    # Local quality is checked at each stage; final cumulative teacher agreement checked on audit.
    FINAL_PASS=bool(all_six and stage_quality and cumulative_delta<=MAX_DELTA_NLL
                    and STUDENT_AUDIT['teacher_agreement']>=MIN_AGREEMENT
                    and FINAL_TIMING['speedup']>MIN_SPEEDUP)
    FINAL=dict(architecture='Qwen3.5-0.8B progressively replaced six full-attention layers with CeNN V5.3',
               model_revision=REVISION,repo_sha=GIT_SHA,source_sha256=SOURCE_SHA256,
               replacement_mode=REPLACEMENT_MODE,exact_groups=EXACT_GROUPS,
               targets=TARGET_LAYERS,installed_layers=list(MANAGER.installed_layers),
               forced_all_six=FORCE_ALL_SIX,all_six_installed=all_six,
               all_stage_quality_pass=stage_quality,final_pass=FINAL_PASS,
               native_validation=NATIVE_DEV,stage_results=REPORT,
               native_audit=NATIVE_AUDIT,student_audit=STUDENT_AUDIT,
               audit_delta_nll=cumulative_delta,paired_audit=FINAL_TIMING,
               audit_source='Disjoint WikiText-2 TRAIN tail, not external blinded holdout',
               data_hashes=DATA_HASHES,run_fingerprint=RUN_FINGERPRINT,
               runtime_seconds=time.time()-RUN_START,
               limitations='18 original gated-delta layers unchanged. No hybrid KV cache, padded batches, '
                           'autoregressive generation, fused recurrence, or independent downstream replication.')
    (OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))
    log_line('FINAL AUDIT '+json.dumps({k:FINAL[k] for k in (
        'all_six_installed','all_stage_quality_pass','final_pass','audit_delta_nll','paired_audit')},
        allow_nan=False))
    if not FINAL_PASS:
        log_line('REJECT as deployment improvement; diagnostic all-six CeNN results retained.')
    RUN_OK=True
except BaseException as exc:
    RUN_ERROR=repr(exc)
    log_line('EXPERIMENT FAILED (partial artifacts preserved): '+RUN_ERROR)
    with (OUT/'error_traceback.txt').open('w') as f:traceback.print_exc(file=f)
    raise
finally:
    # This block runs on complete training and on in-training exceptions.
    pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
    status=dict(completed=RUN_OK,error=RUN_ERROR,installed_layers=list(MANAGER.installed_layers),
                run_fingerprint=RUN_FINGERPRINT,runtime_seconds=time.time()-RUN_START)
    (OUT/'run_status.json').write_text(json.dumps(status,indent=2,allow_nan=False))
    if REPORT:
        try:
            frame=pd.DataFrame(REPORT)
            fig,ax=plt.subplots(figsize=(10,4))
            ax.plot(frame.layer,frame.cumulative_dev_agreement,marker='o',label='Teacher agreement')
            ax.axhline(MIN_AGREEMENT,ls='--',color='gray',label='0.98 quality gate')
            ax.set(xlabel='Replaced full-attention layer',ylabel='Agreement',
                   title='Cumulative six-layer replacement')
            ax.legend();fig.tight_layout()
            fig.savefig(OUT/'cumulative_agreement.png',dpi=160)
            plt.close(fig)
        except Exception as plot_error:
            log_line('Plot not generated: '+repr(plot_error))
    manifest=[]
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.name!='artifact_manifest.json':
            manifest.append(dict(path=str(path.relative_to(OUT)),bytes=path.stat().st_size,
                sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    (OUT/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2))
    archive_path=OUT.parent/(OUT.name+'.zip')
    with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for path in sorted(OUT.rglob('*')):
            if path.is_file():z.write(path,arcname=str(path.relative_to(OUT)))
    ARCHIVE=str(archive_path)
    print('Results ZIP:',ARCHIVE,'bytes:',archive_path.stat().st_size,flush=True)
    print('Includes training.log, training_steps.jsonl, cumulative_layer_results.csv, '
          'final_audit.json (if complete), per-layer checkpoints and manifest.',flush=True)
    if AUTO_DOWNLOAD_RESULTS:
        try:
            from google.colab import files
            files.download(ARCHIVE)  # automatic browser download at end of training/audit
        except ImportError:
            from IPython.display import FileLink,display
            display(FileLink(ARCHIVE))
        except Exception as download_error:
            print('Browser auto-download was blocked:',download_error,
                  '\nRetrieve ZIP from Colab Files panel:',ARCHIVE,flush=True)


### Interpreting the result

`all_six_installed=True` proves that the experiment patched the six full-attention layers, **not** that the substituted model passed accuracy or speed thresholds. Evaluate `final_pass`, `student_audit`, `paired_audit`, each stage's `quality_pass`, and the cumulative NLL/accuracy before considering deployment. Choose `REPLACEMENT_MODE='both_groups'` for replacing every native full-attention head, with expected greater accuracy risk. The other 18 layers remain original. The ZIP download is triggered at the end of the final execution cell, with diagnostic outputs and training logs even on a training-time exception.